# Create retriever training data

In [1]:
import json
import random

def get_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [5]:
chunk_store = get_jsonl_data('../data/chunk_store.jsonl')
print(chunk_store[0])

corpus = get_jsonl_data('../data/corpus.jsonl')
print(corpus[0])

{'doc_id': 'doc_0', 'article_id': 'article_0', 'chunk_id': 0, 'text': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản'}
{'doc_id': 'doc_0', 'article_id': 'article_0', 'article_content': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản\n\n1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng; tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.\n\n2. Tranh chấp quốc tế về địa

In [3]:
retriever_data = []
num_of_negatives = 10

# Group chunks by document to support document-level random negative sampling.
chunks_by_doc = {}
for chunk in chunk_store:
  chunks_by_doc.setdefault(chunk["doc_id"], []).append(chunk["text"])

for i, row in enumerate(corpus):
  # Get positive and negative chunks for the current document
  positive_chunks = [chunk["text"] for chunk in chunk_store if chunk['article_id'].split('_')[1] == str(i)]
  doc_id = row['doc_id']
  negatives_chunks = [
    chunk["text"] for chunk in chunk_store
    if chunk["doc_id"] == doc_id and chunk['article_id'].split('_')[1] != str(i)
  ][:num_of_negatives]

  # If needed, sample negatives from other docs in a document-balanced random way.
  if len(negatives_chunks) < num_of_negatives:
    needed = num_of_negatives - len(negatives_chunks)
    other_doc_ids = [d for d in chunks_by_doc.keys() if d != doc_id and len(chunks_by_doc[d]) > 0]
    random.shuffle(other_doc_ids)

    # Shuffle chunks inside each doc once to avoid always taking top chunks.
    remaining_chunks_by_doc = {
      d: random.sample(chunks_by_doc[d], len(chunks_by_doc[d])) for d in other_doc_ids
    }

    while needed > 0 and other_doc_ids:
      added_in_round = 0
      for other_doc_id in other_doc_ids:
        if not remaining_chunks_by_doc[other_doc_id]:
          continue
        negatives_chunks.append(remaining_chunks_by_doc[other_doc_id].pop())
        needed -= 1
        added_in_round += 1
        if needed == 0:
          break
      if added_in_round == 0:
        break

  qa_pairs = row['generated_qa_pairs']
  for qa_pair in qa_pairs:
    # Append the positive and negative to the retriever_data list
    sample = {
      'question': qa_pair['question'],
      'positive_chunks': positive_chunks,
      'negative_chunks': negatives_chunks
    }
    retriever_data.append(sample)

print(retriever_data[0])

{'question': 'Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong những hoạt động cụ thể nào?', 'positive_chunks': ['Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản', '1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng;', 'tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.', '2. Tranh chấp quốc tế về địa chất, khoáng sản được giải q

In [4]:
count = 0
for sample in retriever_data:
  if sample["negative_chunks"] == []:
    count += 1

print(count)

0


In [ ]:
from pathlib import Path
import json

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

retriever_data_jsonl_path = output_dir / "retriever_data.jsonl"

# Save only the reusable fields as one JSON record per line.
with retriever_data_jsonl_path.open("w", encoding="utf-8") as f:
    for row in retriever_data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(retriever_data)} sample(s) to {retriever_data_jsonl_path}")

Saved 29145 sample(s) to processed\retriever_data.jsonl
